# Street Scout — LoRA Ablation: Base-CLIP-Only Embedding Build

This is a variant of `colab_build_embeddings.ipynb` that skips the `clip_lora` adapter
entirely, using base CLIP ViT-L/14 only. It produces `reference_embeddings_no_lora.pt`
(a separate file, doesn't touch your production `reference_embeddings.pt`) so you can
train a second classifier on it and compare directly against the LoRA-adapted baseline
(currently 65.4% top-1 / 86.2% top-5 on the held-out set) to see whether the LoRA
adapter is actually earning its keep.

Reuses the same files `colab_build_embeddings.ipynb` uses — `MyDrive/street_scout/
reference_images_full.zip`, `class_labels.json`, `reference_images.json`,
`no_car_images.txt` — no new upload needed. Just make sure `no_car_images.txt` in
Drive is the current version if you've updated it locally (it's now 715 entries,
up from 175, after a reference-image quality audit) — both notebooks read it the
same way, so a stale copy in Drive means this run won't reflect that fix either.

Use **Runtime → Change runtime type → T4 GPU** before starting.

In [ ]:
# Step 1 — check we have a GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Step 2 — mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 3 — unzip the full reference image archive from Drive (~12k images, a
# couple of minutes). Unzipping happens here on Linux, which has no issue with
# filenames containing characters like ':' that are problematic on Windows.
# (Matches colab_build_embeddings.ipynb's current approach — the old
# copytree()-from-a-loose-folder + delta-zip method this notebook used
# earlier is stale now that the dataset moved to a single full-zip upload.)
import zipfile, os

DRIVE_DIR = '/content/drive/MyDrive/street_scout'   # class_labels.json, reference_images.json, the zip, etc.
IMAGES_DIR = '/content/reference_images'

full_zip = f'{DRIVE_DIR}/reference_images_full.zip'
os.makedirs(IMAGES_DIR, exist_ok=True)
print('Extracting reference_images_full.zip (this can take a few minutes)...')
with zipfile.ZipFile(full_zip, 'r') as z:
    z.extractall(IMAGES_DIR)

count = len([f for f in os.listdir(IMAGES_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))])
print(f'Extracted {count} images to {IMAGES_DIR}')

In [ ]:
# Step 4 — copy JSON files and fix Windows paths → Colab paths
import json, shutil
from pathlib import PureWindowsPath

shutil.copy(f'{DRIVE_DIR}/class_labels.json', '/content/class_labels.json')
shutil.copy(f'{DRIVE_DIR}/reference_images.json', '/content/reference_images.json')

with open('/content/reference_images.json') as f:
    ref = json.load(f)

fixed = {}
for key, paths in ref.items():
    path_list = paths if isinstance(paths, list) else [paths]
    fixed[key] = [f'{IMAGES_DIR}/{PureWindowsPath(p).name}' for p in path_list]

with open('/content/reference_images.json', 'w') as f:
    json.dump(fixed, f)
print('Path rewrite done.')

no_car_src = f'{DRIVE_DIR}/no_car_images.txt'
NO_CAR_PATH = '/content/no_car_images.txt'
if os.path.exists(no_car_src):
    shutil.copy(no_car_src, NO_CAR_PATH)
    print('no_car_images.txt copied.')
else:
    open(NO_CAR_PATH, 'w').close()
    print('no_car_images.txt not found in Drive — continuing without it.')

In [ ]:
# Step 5 — augmentation + helper logic (same as the main notebook)
from pathlib import Path
from PIL import Image, ImageEnhance

_AUG_VERSION = 3
_MIN_IMAGES_FOR_HOLDOUT = 4
_EMBED_BATCH_SIZE = 64
_CAR_CLASSES = {2, 5, 7}

def _augment(image):
    return [
        image,
        image.transpose(Image.FLIP_LEFT_RIGHT),
        ImageEnhance.Brightness(image).enhance(0.85),
        ImageEnhance.Brightness(image).enhance(1.15),
        image.rotate(-8, expand=False, fillcolor=(128, 128, 128)),
        image.rotate(8, expand=False, fillcolor=(128, 128, 128)),
    ]

def _load_no_car_filenames(path):
    p = Path(path)
    if not p.exists():
        return set()
    return {line.strip() for line in p.read_text(encoding='utf-8').splitlines() if line.strip()}

In [ ]:
# Step 6 — load YOLO (for cropping) and BASE CLIP ONLY (no LoRA) on GPU
from ultralytics import YOLO
from transformers import CLIPModel, CLIPProcessor
import torch.nn.functional as F
import numpy as np

_CLIP_MODEL = 'openai/clip-vit-large-patch14'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

yolo = YOLO('yolov8n.pt')

processor = CLIPProcessor.from_pretrained(_CLIP_MODEL)
clip_model = CLIPModel.from_pretrained(_CLIP_MODEL)   # base CLIP, LoRA intentionally NOT applied
clip_model.eval()
clip_model.to(device)
print('Loaded base CLIP (no LoRA adapter).')

def crop_car(image):
    results = yolo(image, verbose=False)
    boxes = [b for b in results[0].boxes if int(b.cls[0]) in _CAR_CLASSES]
    if not boxes:
        return image
    best = max(boxes, key=lambda b: (b.xyxy[0][2] - b.xyxy[0][0]) * (b.xyxy[0][3] - b.xyxy[0][1]))
    x1, y1, x2, y2 = (int(v) for v in best.xyxy[0])
    return image.crop((x1, y1, x2, y2))

def clip_embed_batch(images):
    enc = processor(images=images, return_tensors='pt')
    pixel_values = enc['pixel_values'].to(device)
    with torch.no_grad():
        out = clip_model.vision_model(pixel_values=pixel_values)
        feat = clip_model.visual_projection(out.pooler_output)
    vecs = F.normalize(feat, dim=-1).cpu().numpy()
    return vecs.astype(np.float32)

In [ ]:
# Step 7 — reserve held-out test images (same deterministic logic as the main
# notebook — will pick the same images, so no need to re-download held_out_test_set.json),
# filter no-car images, then batch-embed everything else
import json, time

with open('/content/class_labels.json', encoding='utf-8') as f:
    labels = json.load(f)
with open('/content/reference_images.json', encoding='utf-8') as f:
    ref_map = json.load(f)

no_car_names = _load_no_car_filenames('/content/no_car_images.txt')

per_key_paths = {}
for key, img_paths in ref_map.items():
    if key not in labels:
        continue
    valid = [Path(p) for p in img_paths if Path(p).exists() and Path(p).name not in no_car_names]
    if valid:
        per_key_paths[key] = valid

held_out_map = {}
for key, paths in per_key_paths.items():
    if len(paths) < _MIN_IMAGES_FOR_HOLDOUT:
        continue
    held_out_path = sorted(paths, key=lambda p: p.name)[-1]
    held_out_map[key] = str(held_out_path)
    per_key_paths[key] = [p for p in paths if p != held_out_path]

print(f'Reserved {len(held_out_map)} held-out test images (should match the LoRA run).')

entries = [(key, p) for key, paths in per_key_paths.items() for p in paths]
n_augs = len(_augment(Image.new('RGB', (4, 4))))
print(f'Embedding {len(entries)} source images x {n_augs} augmentations …')

keys, source_ids, vecs = [], [], []
pending_imgs, pending_keys, pending_sources = [], [], []

def flush():
    if not pending_imgs:
        return
    batch_vecs = clip_embed_batch(pending_imgs)
    keys.extend(pending_keys)
    source_ids.extend(pending_sources)
    vecs.extend(batch_vecs)
    pending_imgs.clear(); pending_keys.clear(); pending_sources.clear()

t0 = time.time()
for i, (key, p) in enumerate(entries):
    try:
        img = Image.open(p).convert('RGB')
        cropped = crop_car(img)
        for aug in _augment(cropped):
            pending_imgs.append(aug)
            pending_keys.append(key)
            pending_sources.append(str(p))
            if len(pending_imgs) >= _EMBED_BATCH_SIZE:
                flush()
    except Exception as exc:
        print(f'Skipping {p}: {exc}')
    if (i + 1) % 500 == 0:
        elapsed = time.time() - t0
        print(f'  …{i+1}/{len(entries)} source images ({elapsed:.0f}s elapsed)')
flush()

emb_matrix = np.stack(vecs).astype(np.float32)
torch.save({
    'keys': keys,
    'embeddings': emb_matrix,
    'source_ids': source_ids,
    'source_count': len(entries),
    'aug_version': _AUG_VERSION,
}, '/content/reference_embeddings_no_lora.pt')
print(f'Done in {time.time()-t0:.0f}s. Saved {len(keys)} embeddings.')

In [ ]:
# Step 8 — copy result back to Google Drive for download
import shutil
shutil.copy('/content/reference_embeddings_no_lora.pt', f'{DRIVE_DIR}/reference_embeddings_no_lora.pt')
print('Saved to Drive:', f'{DRIVE_DIR}/reference_embeddings_no_lora.pt')

## After running

1. Download `reference_embeddings_no_lora.pt` from `MyDrive/street_scout/`.
2. Place it at `ai-service/model/reference_embeddings_no_lora.pt` (a new file — this
   does NOT touch your production `reference_embeddings.pt`).
3. Locally, train a separate classifier on it:
   ```
   python train_classifier.py --cache model/reference_embeddings_no_lora.pt --output model/classifier_no_lora.pkl
   ```
4. Evaluate it against the same held-out set, with LoRA disabled to match:
   ```
   python evaluate.py --no-lora --classifier model/classifier_no_lora.pkl --cache model/reference_embeddings_no_lora.pt --label no_lora
   ```
5. Compare its printed top-1/top-5 against the LoRA baseline (65.4% / 86.2%).